# Helios v0.1.14.1 Portfolio Analysis

Load `equity.csv` / `trades.csv` / `decisions.csv` produced by:
```bash
uv run python scripts/run_portfolio_backtest.py --is-end 2023-12-31 \
    --export-equity equity.csv \
    --export-trades trades.csv \
    --export-decisions decisions.csv
```

Place this notebook in same directory as the 3 CSVs and run cells top-to-bottom.

Required: `polars`, `matplotlib`. Optional: `seaborn` for nicer styling.

In [ ]:

import matplotlib.pyplot as plt
import polars as pl

# Configure plotting
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Adjust paths if your CSVs are elsewhere
EQUITY_CSV = 'equity.csv'
TRADES_CSV = 'trades.csv'
DECISIONS_CSV = 'decisions.csv'

# IS/OOS split (matches run_portfolio_backtest.py --is-end)
IS_END = '2023-12-31'
INITIAL_CAPITAL = 1_000_000

equity = pl.read_csv(EQUITY_CSV, try_parse_dates=True)
trades = pl.read_csv(TRADES_CSV, try_parse_dates=True)
decisions = pl.read_csv(DECISIONS_CSV, try_parse_dates=True)

print(f'Equity curve:  {len(equity)} daily snapshots')
print(f'Trades:        {len(trades)} closed positions')
print(f'Decisions:     {len(decisions)} signal evaluations')
print(f'Date range:    {equity["date"].min()} → {equity["date"].max()}')
print(f'Final equity:  NTD {equity["equity"][-1]:,.0f}')
print(f'Total return:  {(equity["equity"][-1]/INITIAL_CAPITAL-1)*100:+.2f}%')

## 1. Equity curve + IS/OOS split

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

dates = equity['date'].to_list()
eq = equity['equity'].to_list()

ax.plot(dates, eq, linewidth=1.5, color='#2c5aa0', label='Portfolio equity')
ax.axhline(INITIAL_CAPITAL, color='gray', linestyle='--', alpha=0.5, label=f'Initial NTD {INITIAL_CAPITAL:,.0f}')

is_end_date = pl.Series([IS_END]).str.to_date()[0]
ax.axvline(is_end_date, color='#d62728', linestyle=':', alpha=0.7, label=f'IS/OOS split ({IS_END})')

ax.set_title('Helios v0.1.14.1 — Portfolio Equity Curve', fontsize=14)
ax.set_ylabel('Equity (NTD)')
ax.legend(loc='upper left')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e3:,.0f}K'))
plt.tight_layout()
plt.show()

## 2. Drawdown curve (running peak vs current equity)

In [ ]:
eq_df = equity.with_columns(
    pl.col('equity').cum_max().alias('peak')
).with_columns(
    ((pl.col('equity') - pl.col('peak')) / pl.col('peak') * 100).alias('drawdown_pct')
)

max_dd_row = eq_df.sort('drawdown_pct').head(1)
max_dd_pct = max_dd_row['drawdown_pct'][0]
max_dd_date = max_dd_row['date'][0]
print(f'Max drawdown: {max_dd_pct:.2f}% on {max_dd_date}')

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(eq_df['date'].to_list(), eq_df['drawdown_pct'].to_list(), 0,
                color='#d62728', alpha=0.5)
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(is_end_date, color='black', linestyle=':', alpha=0.5)
ax.set_title('Drawdown Curve (peak-to-trough)')
ax.set_ylabel('Drawdown (%)')
ax.annotate(f'  Max DD {max_dd_pct:.1f}%', xy=(max_dd_date, max_dd_pct),
            xytext=(10, 5), textcoords='offset points', fontsize=10)
plt.tight_layout()
plt.show()

## 3. Exposure profile (cash vs deployed over time)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Top: exposure %
axes[0].plot(equity['date'].to_list(), equity['exposure_pct'].to_list(),
             color='#2ca02c', linewidth=1)
axes[0].fill_between(equity['date'].to_list(), equity['exposure_pct'].to_list(), 0,
                     color='#2ca02c', alpha=0.2)
axes[0].axhline(equity['exposure_pct'].mean(), color='red', linestyle='--', alpha=0.5,
                label=f'Avg {equity["exposure_pct"].mean():.1f}%')
axes[0].set_ylabel('Exposure (%)')
axes[0].set_title('Capital Deployment Profile')
axes[0].legend()
axes[0].axvline(is_end_date, color='black', linestyle=':', alpha=0.5)

# Bottom: # positions over time
axes[1].step(equity['date'].to_list(), equity['n_positions'].to_list(),
             color='#ff7f0e', linewidth=1, where='post')
axes[1].set_ylabel('# Positions')
axes[1].set_xlabel('Date')
axes[1].axvline(is_end_date, color='black', linestyle=':', alpha=0.5)
axes[1].set_yticks(range(0, equity['n_positions'].max()+1))

plt.tight_layout()
plt.show()

# Stats
print('\nExposure stats:')
print(f'  Mean:    {equity["exposure_pct"].mean():>5.1f}%')
print(f'  Median:  {equity["exposure_pct"].median():>5.1f}%')
print(f'  Max:     {equity["exposure_pct"].max():>5.1f}%')
print(f'  Days at 0% (fully cash): {(equity["exposure_pct"]==0).sum()} / {len(equity)} ({(equity["exposure_pct"]==0).sum()/len(equity)*100:.1f}%)')

## 4. Trade-level analysis: returns histogram + right-skew confirmation

In [ ]:
returns = trades['gross_return_pct'].to_list()
mean_r = sum(returns) / len(returns)
median_r = sorted(returns)[len(returns)//2]

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(returns, bins=25, color='#2c5aa0', alpha=0.7, edgecolor='black')
ax.axvline(0, color='black', linewidth=1)
ax.axvline(mean_r, color='red', linestyle='--', label=f'Mean {mean_r:+.2f}%')
ax.axvline(median_r, color='orange', linestyle='--', label=f'Median {median_r:+.2f}%')
ax.set_xlabel('Trade Return (%, gross)')
ax.set_ylabel('Frequency')
ax.set_title(f'Trade Returns Distribution (n={len(returns)})  —  Right-skew if mean > median')
ax.legend()
plt.tight_layout()
plt.show()

skew = (mean_r - median_r)
print(f'\nRight-skew indicator: mean - median = {skew:+.2f}% '
      f'({"✓ right-skewed" if skew > 0 else "⚠ left-skewed"})')

## 5. Trade attribution: by sector, exit reason, and regime

In [ ]:
print('=== By sector ===')
print(trades.group_by('sector').agg([
    pl.len().alias('n'),
    pl.col('gross_return_pct').mean().alias('mean_ret'),
    (pl.col('gross_return_pct') > 0).sum().alias('wins'),
    pl.col('net_pnl_ntd').sum().alias('net_pnl_ntd'),
]).sort('mean_ret', descending=True))

print('\n=== By exit reason ===')
print(trades.group_by('exit_reason').agg([
    pl.len().alias('n'),
    pl.col('gross_return_pct').mean().alias('mean_ret'),
    pl.col('holding_days').mean().alias('avg_hold_days'),
]).sort('n', descending=True))

print('\n=== By entry regime (should be all bull) ===')
print(trades.group_by('regime_at_entry').agg([
    pl.len().alias('n'),
    pl.col('gross_return_pct').mean().alias('mean_ret'),
]))

## 6. Signal decisions: accepted vs rejected

In [ ]:
n_accepted = (decisions['decision'] == 'accepted').sum()
n_rejected = (decisions['decision'] == 'rejected').sum()
print(f'Accepted:  {n_accepted} ({n_accepted/len(decisions)*100:.1f}%)')
print(f'Rejected:  {n_rejected} ({n_rejected/len(decisions)*100:.1f}%)')

reject_dist = (decisions.filter(pl.col('decision')=='rejected')
    .group_by('reject_reason')
    .agg(pl.len().alias('n'))
    .sort('n', descending=True))

fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(reject_dist['reject_reason'].to_list(), reject_dist['n'].to_list(), color='#d62728', alpha=0.7)
ax.set_xlabel('# Signals Rejected')
ax.set_title('Reject Reason Distribution — reveals which constraint binds most')
for i, n in enumerate(reject_dist['n'].to_list()):
    ax.text(n+0.5, i, f'  {n} ({n/n_rejected*100:.1f}%)', va='center')
plt.tight_layout()
plt.show()

print('\nReader takeaway:')
print('  - symbol_already_held high → strategy reposts same symbol (low churn)')
print('  - sector_cap_* high       → portfolio diversification protecting from cluster')
print('  - cash_buffer high        → effective max positions limited (4 not 5)')

## 7. Score-decision matrix: do higher-score signals get more accepted?

In [ ]:
score_buckets = decisions.with_columns(
    pl.when(pl.col('score') >= 0.85).then(pl.lit('≥0.85'))
      .when(pl.col('score') >= 0.75).then(pl.lit('0.75-0.85'))
      .when(pl.col('score') >= 0.65).then(pl.lit('0.65-0.75'))
      .otherwise(pl.lit('<0.65'))
      .alias('score_bucket')
)

print(score_buckets.group_by(['score_bucket', 'decision']).agg(pl.len().alias('n'))
    .pivot(values='n', index='score_bucket', on='decision')
    .fill_null(0)
    .with_columns(
        (pl.col('accepted') / (pl.col('accepted')+pl.col('rejected')) * 100).alias('accept_rate_%')
    ).sort('score_bucket', descending=True))

## 8. MFE / MAE scatter (risk profile, reviewer §50)

In [ ]:
# Note: portfolio_simulator currently doesn't export mfe/mae per trade.
# For this section run scripts/run_backtest.py --export-csv trades_rt.csv and load that instead.
# Below is a placeholder using gross_return_pct vs holding_days.

fig, ax = plt.subplots(figsize=(11, 6))
winners = trades.filter(pl.col('gross_return_pct') > 0)
losers = trades.filter(pl.col('gross_return_pct') <= 0)
ax.scatter(winners['holding_days'].to_list(), winners['gross_return_pct'].to_list(),
           color='#2ca02c', alpha=0.6, label=f'Winners (n={len(winners)})')
ax.scatter(losers['holding_days'].to_list(), losers['gross_return_pct'].to_list(),
           color='#d62728', alpha=0.6, label=f'Losers (n={len(losers)})')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Holding days')
ax.set_ylabel('Gross return (%)')
ax.set_title('Trade Outcome vs Holding Period')
ax.legend()
plt.tight_layout()
plt.show()

## Summary

After running all cells, the key questions to answer in your head:

1. **Does the equity curve grow monotonically or is it choppy?** Trend systems are choppy in IS bear years (2022), smooth in OOS bull (2024-2026).
2. **Is the max DD acceptable given your risk tolerance?** -11% on 100 萬 NTD = ~-11 萬 NTD intraday paper loss at worst.
3. **Is 29% avg exposure too lazy or appropriately conservative?** Compare to peace of mind: more exposure = more sleepless nights.
4. **Which sector has the strongest mean return?** ETF > electronics > financial > semi > telecom. Tells you universe composition for v0.2.
5. **Which constraint binds hardest?** Drives next-iteration risk_budget tuning.
6. **Do high-score signals see higher acceptance rate?** They should — selector sorts by score DESC.